<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 5

*Fine-tune CLIP.*

## Imports and initializations

In [ ]:
import gc
import torch
import requests
import numpy as np
import torchvision
import transformers
from PIL import Image
import torch.nn.functional as F
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from collections import Counter
from torchvision import transforms
from sklearn import metrics, preprocessing
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

In [ ]:
BATCH_SIZE = 128

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to('cuda')
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
imagenette = torchvision.datasets.Imagenette('imagenette/', download=True)
imagenette_train, imagenette_test = torch.utils.data.random_split(imagenette, [0.75, 0.25])

train_loader = torch.utils.data.DataLoader(imagenette_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))
test_loader = torch.utils.data.DataLoader(imagenette_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))

In [ ]:
imagenette.classes

## Bare CLIP perfomance

In [ ]:
classes_for_clip = ['A picture of ' + i[0] for i in imagenette.classes]
correct_classifications = 0
all_classifications = 0
predictions = []
truth = []

txt_inputs = tokenizer(text=classes_for_clip, padding=True, return_tensors="pt").to('cuda')
text_features = model.get_text_features(**txt_inputs)
text_emb = text_features.pooler_output.detach().to('cuda')
text_emb = torch.div(text_emb, torch.norm(text_emb, 2, dim=1).repeat(512, 1).T)

for batch, t in tqdm(test_loader):
  with torch.inference_mode():
    img_inputs = processor(images=batch, padding=True, return_tensors='pt').to('cuda')
    image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to('cuda')
    image_emb /= torch.norm(image_emb, 2, dim=1, keepdim=True)

    logits = image_emb @ text_emb.T
    predicted_cat = logits.argmax(dim=1).to('cpu')

    batch_truth = torch.Tensor(t).to('cpu')

    predictions += predicted_cat.tolist()
    truth += t

    correct_classifications += int((predicted_cat == batch_truth).sum())
    all_classifications += BATCH_SIZE

In [ ]:
true_freqs = Counter(truth)
predicted_freqs = Counter(predictions)

fig, ax = plt.subplots(1, 2)
fig.set_figwidth(15)
fig.suptitle('Imagenette class sizes')
ax[0].bar(true_freqs.keys(), true_freqs.values())
ax[1].bar(predicted_freqs.keys(), predicted_freqs.values())

In [ ]:
print(f'Accuracy    {metrics.accuracy_score(truth, predictions)}')
print(f'Precision   {metrics.precision_score(truth, predictions, average='macro')}')
print(f'Recall      {metrics.recall_score(truth, predictions, average='macro')}')
print(f'F1          {metrics.f1_score(truth, predictions, average='macro')}')

## Adding a layer after CLIP image processor

In [ ]:
linear_layer = torch.nn.Linear(in_features=512, out_features=10, device='cuda')
optimizer = torch.optim.Adam(linear_layer.parameters())
loss = torch.nn.CrossEntropyLoss()

### Training

In [ ]:
epochs = 10
for epoch in range(epochs):
  epoch_loss =  0
  for batch, t in tqdm(train_loader):
    img_inputs = processor(images=batch, padding=True, return_tensors='pt').to('cuda')
    image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to('cuda')
    image_emb /= torch.norm(image_emb, 2, dim=1, keepdim=True)

    optimizer.zero_grad()
    logits = linear_layer.forward(image_emb)
    loss_val = loss(logits, torch.tensor(t).to('cuda'))
    loss_val.backward()
    optimizer.step()
    epoch_loss += loss_val.item()
  print(f'Loss epoch {epoch}: {epoch_loss / train_loader.batch_size}')

### Evaluating

In [ ]:
classes_for_clip = ['A picture of ' + i[0] for i in imagenette.classes]
correct_classifications = 0
all_classifications = 0
predictions = []
truth = []

txt_inputs = tokenizer(text=classes_for_clip, padding=True, return_tensors="pt").to('cuda')
text_features = model.get_text_features(**txt_inputs)
text_emb = text_features.pooler_output.detach().to('cuda')
text_emb = torch.div(text_emb, torch.norm(text_emb, 2, dim=1).repeat(512, 1).T)

for batch, t in tqdm(test_loader):
  with torch.inference_mode():
    img_inputs = processor(images=batch, padding=True, return_tensors='pt').to('cuda')
    image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to('cuda')
    image_emb /= torch.norm(image_emb, 2, dim=1, keepdim=True)

    logits = linear_layer(image_emb) # Instead of logits = image_emb @ text_emb.T
    predicted_cat = logits.argmax(dim=1).to('cpu')

    batch_truth = torch.Tensor(t).to('cpu')

    predictions += predicted_cat.tolist()
    truth += t

    correct_classifications += int((predicted_cat == batch_truth).sum())
    all_classifications += BATCH_SIZE

In [ ]:
true_freqs = Counter(truth)
predicted_freqs = Counter(predictions)

fig, ax = plt.subplots(1, 2)
fig.set_figwidth(15)
fig.suptitle('Imagenette class sizes')
ax[0].bar(true_freqs.keys(), true_freqs.values())
ax[1].bar(predicted_freqs.keys(), predicted_freqs.values())

print(f'Accuracy    {metrics.accuracy_score(truth, predictions)}')
print(f'Precision   {metrics.precision_score(truth, predictions, average='macro')}')
print(f'Recall      {metrics.recall_score(truth, predictions, average='macro')}')
print(f'F1          {metrics.f1_score(truth, predictions, average='macro')}')
print()